In [ ]:
#!/usr/bin/env python3
"""
Publishes the current time every 10 seconds.
QoS 1 ensures the broker queues messages for offline persistent-session subscribers.

Broker options (change BROKER/PORT below):
  Local:  localhost:1883   (brew install mosquitto && brew services start mosquitto)
  Docker: localhost:1883   (docker run -d -p 1883:1883 eclipse-mosquitto)
  Public: test.mosquitto.org  (unreliable for persistent session queuing)
"""

import time
from datetime import datetime
import paho.mqtt.client as mqtt

BROKER = "localhost"
PORT = 1883
TOPIC = "apoorv/time/publish"
INTERVAL = 10  # seconds
QOS = 1


def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code == 0:
        print(f"Connected to {BROKER}")
    else:
        print(f"Connection failed with code {reason_code}")


def on_publish(client, userdata, mid, reason_code, properties):
    print(f"  Message {mid} acknowledged by broker")


client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_connect = on_connect
client.on_publish = on_publish

client.connect(BROKER, PORT, keepalive=60)
client.loop_start()

print(f"Publishing current time to '{TOPIC}' every {INTERVAL}s. Press Ctrl+C to stop.\n")
i = 0
try:
    while True:
        now = datetime.now().isoformat()
        message = f"Message {i} {now}"
        i += 1
        result = client.publish(TOPIC, message, qos=QOS)
        print(f"[{now}] Published: {message}  (mid={result.mid})")
        time.sleep(INTERVAL)
except KeyboardInterrupt:
    print("\nStopped.")
finally:
    client.loop_stop()
    client.disconnect()
